In [ ]:
import networkx as nx
import heapq
import time
from scipy.io import mmread
import matplotlib.pyplot as plt




==========================
Load Graph
==========================


In [ ]:

def load_graph(file_path):

    if file_path.endswith(".mtx"):

        matrix = mmread(file_path)

        G = nx.from_scipy_sparse_array(matrix)

    elif file_path.endswith(".edges"):

        G = nx.Graph()

        with open(file_path, "r") as file:

            for line in file:

                if line.startswith("%"):
                    continue

                parts = line.split()

                if len(parts) >= 2:

                    u = int(parts[0])
                    v = int(parts[1])

                    G.add_edge(u, v)

    else:
        raise ValueError(f"Unsupported file format: {file_path}")

    return G




==========================
A* Algorithm
==========================


In [ ]:

def heuristic(u, v):
    return 0   # no coordinates available


def a_star(G, source, target):

    open_set = [(0, source)]

    g_score = {node: float('inf') for node in G.nodes()}
    g_score[source] = 0

    parent = {source: None}

    visited_order = []

    while open_set:

        f_current, current = heapq.heappop(open_set)

        if current in visited_order:
            continue

        visited_order.append(current)

        if current == target:
            break

        for neighbor in G.neighbors(current):

            edge_weight = G[current][neighbor].get("weight", 1)

            tentative_g = g_score[current] + edge_weight

            if tentative_g < g_score[neighbor]:

                g_score[neighbor] = tentative_g
                parent[neighbor] = current

                f_score = tentative_g + heuristic(neighbor, target)

                heapq.heappush(open_set, (f_score, neighbor))

    return g_score, parent, visited_order




==========================
DATASETS
==========================


In [ ]:

datasets = [
    "inf-USAir97.mtx",
    "inf-power.mtx",
    "road-roadNet-CA.mtx",
    "road-euroroad.edges"
]




==========================
MAIN
==========================


In [ ]:

if __name__ == "__main__":

    print("\n===== A* RESULTS =====\n")

    dataset_names = []
    execution_times = []
    node_counts = []
    edge_counts = []

    for dataset in datasets:

        print("=" * 60)
        print("Dataset:", dataset)

        G = load_graph(dataset)

        print("Nodes:", G.number_of_nodes())
        print("Edges:", G.number_of_edges())

        source = list(G.nodes())[0]
        target = list(G.nodes())[-1]

        start = time.perf_counter()

        distances, parent, order = a_star(G, source, target)

        end = time.perf_counter()

        execution_time = end - start

        print("Execution Time:", round(execution_time, 6), "seconds")
        print("Visited Nodes:", len(order))
        print()

        dataset_names.append(dataset)
        execution_times.append(execution_time)
        node_counts.append(G.number_of_nodes())
        edge_counts.append(G.number_of_edges())




==========================
Execution Time Graph
==========================


In [ ]:

    plt.figure(figsize=(8, 5))

    plt.plot(dataset_names, execution_times, marker="o")

    plt.title("A* Execution Time")

    plt.xlabel("Datasets")
    plt.ylabel("Time (seconds)")
    plt.grid()

    plt.savefig("astar_execution_time.png")
    plt.show()




==========================
Node Growth
==========================


In [ ]:

    plt.figure(figsize=(8, 5))

    plt.bar(dataset_names, node_counts)

    plt.title("Node Growth (A*)")

    plt.xlabel("Datasets")
    plt.ylabel("Nodes")
    plt.grid()

    plt.savefig("astar_node_growth.png")
    plt.show()




==========================
Edge Growth
==========================


In [ ]:

    plt.figure(figsize=(8, 5))

    plt.bar(dataset_names, edge_counts)

    plt.title("Edge Growth (A*)")

    plt.xlabel("Datasets")
    plt.ylabel("Edges")
    plt.grid()

    plt.savefig("astar_edge_growth.png")
    plt.show()
